In [1]:
import time
import os
import psutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from autogluon.tabular import TabularDataset, TabularPredictor

In [2]:
df = pd.read_csv("../data/creditcard.csv")

In [3]:
!python -m pip freeze

absl-py==2.5.0
accelerate==1.15.0
adagio==0.2.6
aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiohttp-cors==0.8.1
aiosignal==1.4.0
alembic==1.20.0
annotated-doc==0.0.5
annotated-types==0.8.0
antlr4-python3-runtime==4.9.3
anyio==4.15.1
appnope==1.0.0
apsw==3.53.4.0
apswutils==0.1.2
asttokens==3.0.2
attrs==26.1.0
autogluon==1.6.3
autogluon.common==1.6.3
autogluon.core==1.6.3
autogluon.features==1.6.3
autogluon.multimodal==1.6.3
autogluon.tabular==1.6.3
autogluon.timeseries==1.6.3
beartype==0.22.9
beautifulsoup4==4.15.0
blis==1.3.3
boto3==1.43.105
botocore==1.43.105
catalogue==2.0.10
catboost==1.2.10
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
chronos-forecasting==2.3.2
click==8.5.0
cloudpathlib==0.25.0
cloudpickle==3.1.2
colorama==0.4.6
colorful==0.5.8
colorlog==6.12.0
comm==0.2.3
confection==1.3.3
contourpy==1.4.0
coreforecast==0.0.16
cryptography==50.0.1
cycler==0.12.1
cymem==2.0.13
datasets==5.0.1
debugpy==1.8.22
defusedxml==0.7.1
dill==0.4.1
distlib==0.4.3
einops==0.8.2
einx==

In [4]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# COLUNAS
# =========================
X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)

# =========================
# INSTALAR AUTOGLUON
# =========================
# %pip install autogluon psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    train_data = TabularDataset(train_df)
    test_data = TabularDataset(test_df)

    # =========================
    # AUTOGLUON (TREINAMENTO + MÉTRICA COMPUTACIONAL)
    # =========================
    print("Iniciando treinamento do AutoGluon...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    predictor = TabularPredictor(
        label='Class'
    ).fit(
        train_data=train_data
    )

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    leaderboard = predictor.leaderboard(test_data, display=True)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    resultados = []

    X_test_ag = test_df.drop(columns=['Class'])
    y_true = test_df['Class'].values

    # Iterar sobre todos os modelos gerados (incluindo Ensembles)
    for model_name in predictor.model_names():
        # Tempo de inferência no conjunto de teste (previsão + score)
        inicio_execucao = time.perf_counter()
        
        preds = predictor.predict(X_test_ag, model=model_name)
        y_pred = preds.astype(int).values

        y_score = predictor.predict_proba(X_test_ag, model=model_name, as_multiclass=False).values
        
        tempo_execucao = time.perf_counter() - inicio_execucao

        # Métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        average_precision = round(average_precision_score(y_true, y_score), 4)

        resultados.append({
            "Modelo": model_name,
            "Tipo": model_name.split('_')[0],
            "Seed": SEED,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "F1-score Weighted": f1_weighted,
            "average_precision": average_precision,
            "Tempo de execução no teste (s)": round(tempo_execucao, 4),
            "Tempo Treino Total (s)": tempo_treino_total,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    # =========================
    # RESULTADO FINAL
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS (SEED: {}) =====".format(SEED))
    print(df_resultados.sort_values(by="average_precision", ascending=False).to_string(index=False))


EXECUTANDO COM SEED: 42


No path specified. Models will be saved in: "AutogluonModels/ag-20261106_153019"
Verbosity: 2 (Standard Logging)


Iniciando treinamento do AutoGluon...


=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.63 GB / 16.00 GB (22.7%)
Disk Space Avail:   234.55 GB / 460.38 GB (50.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best'


===== LEADERBOARD =====
                  model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      RandomForestEntr    0.999649   0.999202    accuracy        0.069914       0.026252  27.522594                 0.069914                0.026252          27.522594            1       True          4
1        ExtraTreesGini    0.999649   0.999601    accuracy        0.117586       0.026406   3.178269                 0.117586                0.026406           3.178269            1       True          6
2      RandomForestGini    0.999631   0.999202    accuracy        0.084476       0.026499  40.484435                 0.084476                0.026499          40.484435            1       True          3
3        ExtraTreesEntr    0.999631   0.999601    accuracy        0.107888       0.036489   2.968776                 0.107888                0.036489          

No path specified. Models will be saved in: "AutogluonModels/ag-20261106_153239"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.42 GB / 16.00 GB (21.4%)
Disk Space Avail:   234.44 GB / 460.38 GB (50.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The

... (11 outputs left out)

No path specified. Models will be saved in: "AutogluonModels/ag-20261106_153927"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.84 GB / 16.00 GB (24.0%)
Disk Space Avail:   234.13 GB / 460.38 GB (50.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The


===== MÉTRICAS DE TODOS OS MODELOS (SEED: 302) =====
             Modelo             Tipo  Seed  Accuracy  Precision  Recall  F1-score  F1-score Weighted  average_precision  Tempo de execução no teste (s)  Tempo Treino Total (s)  Uso RAM Treino (MB)
   RandomForestEntr RandomForestEntr   302    0.9996     0.9639  0.8163    0.8840             0.9996             0.8943                          0.1646                  134.73              1029.09
   RandomForestGini RandomForestGini   302    0.9997     0.9647  0.8367    0.8962             0.9997             0.8894                          0.1908                  134.73              1029.09
     ExtraTreesGini   ExtraTreesGini   302    0.9997     0.9759  0.8265    0.8950             0.9997             0.8879                          0.2869                  134.73              1029.09
WeightedEnsemble_L2 WeightedEnsemble   302    0.9997     0.9759  0.8265    0.8950             0.9997             0.8879                          0.2385       

	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 30 | ['Time', 'V1', 'V2', 'V3', 'V4', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 30 | ['Time', 'V1', 'V2', 'V3', 'V4', ...]
	0.2s = Fit runtime
	30 features in original data used to generate 30 features in processed data.
	Train Data (Processed) Memory Usage: 52.15 MB (1.3% of available memory)
Data preprocessing and feature engineering runtime = 0.27s ...
AutoGluon will gauge predictive performance using evaluation metric: 'accuracy'
	To change this, specify the eval_metric parameter of Predictor()
Automatically generating train/validation split with holdout_frac=0


===== LEADERBOARD =====
                  model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0        NeuralNetTorch    0.999508   1.000000    accuracy        0.160326       0.009060  18.461197                 0.160326                0.009060          18.461197            1       True         10
1      RandomForestGini    0.999491   1.000000    accuracy        0.078887       0.028845  48.512016                 0.078887                0.028845          48.512016            1       True          3
2      RandomForestEntr    0.999473   1.000000    accuracy        0.066439       0.031348  30.920506                 0.066439                0.031348          30.920506            1       True          4
3        ExtraTreesEntr    0.999473   1.000000    accuracy        0.114047       0.045268   3.527944                 0.114047                0.045268          